## Setup

### Imports

In [1]:
import anndata as ad
import mudata as md
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

In [2]:
# load precursor and protein data from diann with alphapepttools functions

In [3]:
# load h5ad data saved by alphapepttools
prec_adata = ad.read_h5ad("../data/albrecht.precursors.h5ad")
prot_adata = ad.read_h5ad("../data/albrecht.proteins.h5ad")

In [4]:
msdata = md.MuData(
    # These are the raw data levels
    {
        "protein_level": prot_adata,
        #"peptide_level": ad.AnnData(...),
        "precursor_level": prec_adata,
    },
    # This stores the feature mapping as adjacency matrix of a DAG
    #varp = csr_matrix(...)
)

/Users/mcthielert/miniforge3/envs/msmudata_dev/lib/python3.14/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/Users/mcthielert/miniforge3/envs/msmudata_dev/lib/python3.14/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [5]:
# Build the precursor -> protein adjacency matrix (varp)
n_prec = len(prec_adata.var)
n_prot = len(prot_adata.var)
n_total = n_prec + n_prot 

# Each precursor maps to one protein group via Protein.Group column
prot_index = pd.Index(prot_adata.var.index)
prec_protein_groups = prec_adata.var["Protein.Group"]

# Precursors are rows 0:n_prec, proteins are rows n_prec:n_total
row_idx = np.arange(n_prec)
col_idx = prot_index.get_indexer(prec_protein_groups) + n_prec  # offset into protein block

assert (col_idx >= n_prec).all(), "Some precursors don't map to any protein group!"

mapping_matrix_varp = csr_matrix(
    (np.ones(len(row_idx), dtype=np.float32), (row_idx, col_idx)),
    shape=(n_total, n_total),
)

print(f"Mapping matrix shape: {mapping_matrix_varp.shape}")  
print(f"Non-zero entries: {mapping_matrix_varp.nnz}")     

Mapping matrix shape: (17250, 17250)
Non-zero entries: 15089


In [6]:
msdata

MuData object with n_obs × n_vars = 1801 × 17250
  2 modalities
    protein_level:	1801 x 2161
    precursor_level:	1801 x 15089
      var:	'Run', 'Stripped.Sequence', 'Precursor.Charge', 'RT', 'RT.Start', 'RT.Stop', 'IM', 'Protein.Group', 'Protein.Ids', 'Genes', 'MS2.Scan', 'CScore', 'Q.Value', 'Global.Q.Value', 'Global.PG.Q.Value', 'Lib.Q.Value', 'Lib.PG.Q.Value'

In [7]:
#msdata.varm["precursor_to_protein"] = mapping_matrix_varm
msdata.varp["precursor_to_protein"] = mapping_matrix_varp

In [8]:
msdata

MuData object with n_obs × n_vars = 1801 × 17250
  varp:	'precursor_to_protein'
  2 modalities
    protein_level:	1801 x 2161
    precursor_level:	1801 x 15089
      var:	'Run', 'Stripped.Sequence', 'Precursor.Charge', 'RT', 'RT.Start', 'RT.Stop', 'IM', 'Protein.Group', 'Protein.Ids', 'Genes', 'MS2.Scan', 'CScore', 'Q.Value', 'Global.Q.Value', 'Global.PG.Q.Value', 'Lib.Q.Value', 'Lib.PG.Q.Value'

## implement on functions

In [36]:
def get_unique_mappings(psm_table: pd.DataFrame, feature_level_names: list[str]) -> pd.DataFrame:
    """
    Get unique mappings from PSM table for specified feature levels.
    
    Parameters:
    - psm_table: DataFrame containing PSM data with columns for each feature level.
    - feature_level_names: List of column names corresponding to feature levels (e.g., ["Precursor", "Protein"]).
    
    Returns:
    - DataFrame with unique mappings between the specified feature levels.
    """
    # Select only the relevant columns for mapping
    mapping_df = psm_table[feature_level_names].drop_duplicates()
    
    return mapping_df

In [37]:
test_map = get_unique_mappings(prec_adata.var, ["Protein.Group"])

In [38]:
test_map

,Protein.Group
AAAAAAALQAK2,P36578
AAAATGTIFTFR2,P05154
AAAFEEQENETVVVK2,Q9Y490
AAAFLGDIALDEEDLR3,P13497
AAAGEFADDPC(UniMod:4)SSVK2,P35221;P35221-2
...,...
YPVVPVHLDTTI2,P12724
YQAVTATLEEK2,P40429
YQFFVYLQEGK2,Q96S96
YREWHHFLVVNMK4,P30086


In [ ]:
def sparse_matrix_mapping(
    mapping_df: pd.DataFrame
) -> csr_matrix:
    """
    Create a square sparse adjacency matrix for varp from a feature-level mapping.

    Parameters
    ----------
    mapping_df : pd.DataFrame
        DataFrame containing the mapping between source and targets features.

    Returns
    -------
    csr_matrix
        Square matrix of shape (n_total_vars, n_total_vars).
    """
    source_var = mapping_df.index
    print("Source variables")
    print(source_var)
    
    for col in mapping_df.columns:
        target_var_index = mapping_df[col].unique()
        print("Target variables")
        print(target_var_index)

    mapping_matrix = mapping_df
    
    # Map source features to their global position in msdata.var
    source_global_idx = global_var_names.get_indexer(source_var.index)
    # # Map each source feature's target to the target's global position
    # target_global_idx = global_var_names.get_indexer(target_var_index[
    #     target_var_index.get_indexer(source_var[mapping_col])
    # ])

    # # Validate
    # assert (source_global_idx >= 0).all(), "Some source features not found in MuData var"
    # assert (target_global_idx >= 0).all(), f"Some {mapping_col} values not found in target modality"

    # n_total = len(global_var_names)
    # mapping_matrix = csr_matrix(
    #     (np.ones(len(source_global_idx), dtype=np.float32), (source_global_idx, target_global_idx)),
    #     shape=(n_total, n_total),
    # )
    return mapping_matrix

In [44]:
matrix_varp = sparse_matrix_mapping(test_map)

Source variables
Index(['AAAAAAALQAK2', 'AAAATGTIFTFR2', 'AAAFEEQENETVVVK2',
       'AAAFLGDIALDEEDLR3', 'AAAGEFADDPC(UniMod:4)SSVK2',
       'AAAIGIDLGTTYSC(UniMod:4)VGVFQHGK3', 'AAAPAPVSEAVC(UniMod:4)R2',
       'AAAPNTPK1', 'AAATLMSER2', 'AAATPESQEPQAK2',
       ...
       'YGPVFSFTMVGK2', 'YLNGLGR2', 'YLYTLEK2', 'YLYTLNDNAR2', 'YNLGLDLR2',
       'YPVVPVHLDTTI2', 'YQAVTATLEEK2', 'YQFFVYLQEGK2', 'YREWHHFLVVNMK4',
       'YVLMVVASDR2'],
      dtype='object', length=2161)
Target variables
['P36578', 'P05154', 'Q9Y490', 'P13497', 'P35221;P35221-2', ..., 'P12724', 'P40429', 'Q96S96', 'P30086', 'Q6V0I7;Q6V0I7-3']
Length: 2161
Categories (2161, object): ['A0A0A0MRZ8;P04433', 'A0A0A0MS15', 'A0A0A0MT31', 'A0A0A0MT36', ..., 'S4R471', 'V9GYG9', 'V9GYJ8', 'V9GYS1']


In [32]:
matrix_varp

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1 stored elements and shape (2161, 2161)>

In [ ]:
def sparse_matrix_mapping(
    mapping_df: pd.DataFrame,
    source_modality: str,
    target_modality: str,
    mapping_col: str,
) -> csr_matrix:
    """
    Create a square sparse adjacency matrix for varp from a feature-level mapping.

    Parameters
    ----------
    msdata : MuData
        The MuData object containing both modalities.
    source_modality : str
        Name of the source modality (e.g. "precursor_level").
    target_modality : str
        Name of the target modality (e.g. "protein_level").
    mapping_col : str
        Column in source's .var that maps to target's var index.

    Returns
    -------
    csr_matrix
        Square matrix of shape (n_total_vars, n_total_vars).
    """
    global_var_names = msdata.var_names

    source_var = msdata.mod[source_modality].var
    target_var_index = pd.Index(msdata.mod[target_modality].var.index)

    # Map source features to their global position in msdata.var
    source_global_idx = global_var_names.get_indexer(source_var.index)
    # Map each source feature's target to the target's global position
    target_global_idx = global_var_names.get_indexer(target_var_index[
        target_var_index.get_indexer(source_var[mapping_col])
    ])

    # Validate
    assert (source_global_idx >= 0).all(), "Some source features not found in MuData var"
    assert (target_global_idx >= 0).all(), f"Some {mapping_col} values not found in target modality"

    n_total = len(global_var_names)
    mapping_matrix = csr_matrix(
        (np.ones(len(source_global_idx), dtype=np.float32), (source_global_idx, target_global_idx)),
        shape=(n_total, n_total),
    )
    return mapping_matrix

In [ ]:
def sparse_matrix_mapping(
    mapping_df: pd.DataFrame
) -> csr_matrix:
    """
    Create a square sparse adjacency matrix for varp from a feature-level mapping.

    Parameters
    ----------
    mapping_df : pd.DataFrame
        DataFrame containing the mapping between source and targets features.

    Returns
    -------
    csr_matrix
        Square matrix of shape (n_total_vars, n_total_vars).
    """
    source_var = mapping_df.index
    for col in mapping_df.columns:
        target_var_index = mapping_df[col].unique()
    
    # Map source features to their global position in msdata.var
    source_global_idx = global_var_names.get_indexer(source_var.index)
    # Map each source feature's target to the target's global position
    target_global_idx = global_var_names.get_indexer(target_var_index[
        target_var_index.get_indexer(source_var[mapping_col])
    ])

    # Validate
    assert (source_global_idx >= 0).all(), "Some source features not found in MuData var"
    assert (target_global_idx >= 0).all(), f"Some {mapping_col} values not found in target modality"

    n_total = len(global_var_names)
    mapping_matrix = csr_matrix(
        (np.ones(len(source_global_idx), dtype=np.float32), (source_global_idx, target_global_idx)),
        shape=(n_total, n_total),
    )
    return mapping_matrix

In [21]:
def sparse_matrix_mapping(mapping_df: pd.DataFrame) -> csr_matrix:
    """
    Create a sparse matrix mapping from source to target features.
    
    Parameters:
    - mapping_df: DataFrame containing unique mappings between source and target features.

    Returns:
    - Sparse matrix (csr_matrix) representing the mapping from source to target features.
    """
    # Get unique source and target features
    unique_prec = mapping_df.index.unique()
    unique_prot = mapping_df['Protein.Group'].unique()
    
    # Create index mappings for sources and targets
    source_index = {feature: idx for idx, feature in enumerate(unique_prec)}
    target_index = {feature: idx for idx, feature in enumerate(unique_prot)}
    
    # Prepare data for sparse matrix construction
    row_idx = mapping_df.index.map(target_index).values
    col_idx = mapping_df['Protein.Group'].map(source_index).values

    data = np.ones(len(mapping_df), dtype=np.float32)
    
    # Create sparse matrix
    sparse_matrix = csr_matrix((data, (row_idx, col_idx)), shape=(len(unique_prot), len(unique_prec)))
    
    return sparse_matrix

In [22]:
matrix_varp = sparse_matrix_mapping(test_map)

/Users/mcthielert/miniforge3/envs/msmudata_dev/lib/python3.14/site-packages/scipy/sparse/_coo.py:62: RuntimeWarning: invalid value encountered in cast
  self.coords = tuple(np.array(idx, copy=copy, dtype=idx_dtype)


In [23]:
matrix_varp

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1 stored elements and shape (2161, 2161)>

In [ ]:

# Check for unmapped precursors (0 means no match)
unmapped = prec_adata.var[col_idx == 0]
print(f"Unmapped precursors: {len(unmapped)}")
print(unmapped["Protein.Group"].head())  # inspect why they don't match

# Build CSR: shape (n_prec x n_prot)
row_idx = np.arange(len(prec_adata.var))

mapping_matrix = csr_matrix(
    (np.ones(len(row_idx[col_idx >= 0]), dtype=np.float32),
     (row_idx[col_idx >= 0], col_idx[col_idx >= 0])),
    shape=(len(prec_adata.var), len(prot_adata.var)),
)

In [ ]:
def sparse_matrix_mapping(mapping_df: pd.DataFrame) -> csr_matrix:
    """
    Create a sparse matrix mapping from source to target features.
    
    Parameters:
    - mapping_df: DataFrame containing unique mappings between source and target features.

    Returns:
    - Sparse matrix (csr_matrix) representing the mapping from source to target features.
    """
    # Get unique source and target features
    unique_sources = mapping_df[source_col].unique()
    unique_targets = mapping_df[target_col].unique()
    
    # Create index mappings for sources and targets
    source_index = {feature: idx for idx, feature in enumerate(unique_sources)}
    target_index = {feature: idx for idx, feature in enumerate(unique_targets)}
    
    # Prepare data for sparse matrix construction
    row_idx = mapping_df[source_col].map(source_index).values
    col_idx = mapping_df[target_col].map(target_index).values
    data = np.ones(len(mapping_df), dtype=np.float32)
    
    # Create sparse matrix
    sparse_matrix = csr_matrix((data, (row_idx, col_idx)), shape=(len(unique_sources), len(unique_targets)))
    
    return sparse_matrix

In [17]:
mapping_matrix_varp.indices

array([16069, 15553, 15553, ..., 15477, 15477, 15477],
      shape=(15089,), dtype=int32)

In [18]:
msdata.var_names

Index(['A0A024R6I7;A0A0G2JRN3', 'A0A075B6H7', 'A0A075B6I0', 'A0A075B6I9',
       'A0A075B6J2', 'A0A075B6J9', 'A0A075B6K4', 'A0A075B6K5',
       'A0A075B6P5;P01615', 'A0A075B6Q5',
       ...
       'YYTVFDR2', 'YYTVFDRDNNR2', 'YYTYLIM(UniMod:35)NK2', 'YYTYLIMNK2',
       'YYVTIIDAPGHR2', 'YYVTIIDAPGHR3', 'YYVTIIDAPGHRDFIK4',
       'YYWGGQYTWDM(UniMod:35)AK2', 'YYWGGQYTWDMAK2', 'YYWGGQYTWDMAK3'],
      dtype='object', length=17250)

In [22]:
pd.DataFrame.sparse.from_spmatrix(mapping_matrix_varp, index=msdata.var_names, columns=msdata.var_names).reindex(msdata.var_names,columns=msdata.var_names)

,A0A024R6I7;A0A0G2JRN3,A0A075B6H7,A0A075B6I0,A0A075B6I9,A0A075B6J2,A0A075B6J9,A0A075B6K4,A0A075B6K5,A0A075B6P5;P01615,A0A075B6Q5,...,YYTVFDR2,YYTVFDRDNNR2,YYTYLIM(UniMod:35)NK2,YYTYLIMNK2,YYVTIIDAPGHR2,YYVTIIDAPGHR3,YYVTIIDAPGHRDFIK4,YYWGGQYTWDM(UniMod:35)AK2,YYWGGQYTWDMAK2,YYWGGQYTWDMAK3
A0A024R6I7;A0A0G2JRN3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A0A075B6H7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A0A075B6I0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A0A075B6I9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A0A075B6J2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YYVTIIDAPGHR3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
YYVTIIDAPGHRDFIK4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
YYWGGQYTWDM(UniMod:35)AK2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
YYWGGQYTWDMAK2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
mapping_matrix_varp.sum()

np.float32(15089.0)

In [16]:
prot_adata

AnnData object with n_obs × n_vars = 1801 × 2161

In [17]:
prec_adata

AnnData object with n_obs × n_vars = 1801 × 15089
    var: 'Run', 'Stripped.Sequence', 'Precursor.Charge', 'RT', 'RT.Start', 'RT.Stop', 'IM', 'Protein.Group', 'Protein.Ids', 'Genes', 'MS2.Scan', 'CScore', 'Q.Value', 'Global.Q.Value', 'Global.PG.Q.Value', 'Lib.Q.Value', 'Lib.PG.Q.Value'

In [ ]:
# Build the precursor -> protein adjacency matrix (varp)
# Each precursor maps to one protein group via Protein.Group column
prot_index = pd.Index(prot_adata.var.index)
prec_protein_groups = prec_adata.var["Protein.Group"]


# Row indices: precursors, Col indices: proteins
row_idx = np.arange(len(prec_adata.var))
col_idx = prot_index.get_indexer(prec_protein_groups)

# Sanity check: all precursors should map to a known protein group
assert (col_idx >= 0).all(), "Some precursors don't map to any protein group!"

mapping_matrix = csr_matrix(
    (np.ones(len(row_idx), dtype=np.float32), (row_idx, col_idx)),
    shape=(len(prec_adata.var), len(prot_adata.var)),
)
print(f"Mapping matrix shape: {mapping_matrix.shape} (precursors x proteins)")
print(f"Non-zero entries: {mapping_matrix.nnz}")


NameError: name 'pd' is not defined

In [ ]:

# Row indices: precursors, Col indices: proteins
row_idx = np.arange(len(prec_adata.var))
col_idx = prot_index.get_indexer(prec_protein_groups)

# Sanity check: all precursors should map to a known protein group
assert (col_idx >= 0).all(), "Some precursors don't map to any protein group!"

mapping_matrix = csr_matrix(
    (np.ones(len(row_idx), dtype=np.float32), (row_idx, col_idx)),
    shape=(len(prec_adata.var), len(prot_adata.var)),
)
print(f"Mapping matrix shape: {mapping_matrix.shape} (precursors x proteins)")
print(f"Non-zero entries: {mapping_matrix.nnz}")